# Tema 5 – Ejercicio 2: Transformer básico para detección de Fake News

En este notebook se resuelve el **Ejercicio 2** de la hoja de ejercicios del Tema 5.

El objetivo es entrenar un modelo **encoder de tipo Transformer** para clasificar noticias como:

- `0` → noticia verdadera
- `1` → noticia falsa

Se entrenan dos versiones:

1. **Transformer sin embeddings posicionales**.
2. **Transformer con embeddings posicionales** usando `keras_nlp.layers.PositionEmbedding`.

Al final se comparan los resultados usando **accuracy** y **Macro F1**.

## 1. Instalación de dependencias

Si estás ejecutando este notebook en **Colab**, puede que necesites instalar `keras-nlp`.

Si ya lo tienes instalado, puedes dejar esta celda comentada.

In [ ]:
# Si estás en Colab o no tienes keras-nlp instalado, descomenta esta línea:
# !pip install keras-nlp -q

## 2. Imports

Importamos las librerías necesarias:

- `pandas` y `numpy` para manejar datos.
- `tensorflow` y `keras` para crear el modelo.
- `keras_nlp` para usar embeddings posicionales.
- `sklearn` para dividir el dataset y evaluar resultados.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

try:
    import keras_nlp
    KERAS_NLP_AVAILABLE = True
except ImportError:
    KERAS_NLP_AVAILABLE = False
    print("keras-nlp no está instalado. Instálalo con: pip install keras-nlp")

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

from tensorflow.keras.layers import (
    Input,
    Embedding,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Semillas para hacer los resultados más reproducibles
np.random.seed(42)
tf.random.set_seed(42)

## 3. Carga del dataset

El dataset usado en clase tiene columnas similares a:

- `Headline`: titular de la noticia.
- `Text`: cuerpo de la noticia.
- `Category`: etiqueta de la noticia.

Cambia `CSV_PATH` si tu archivo tiene otro nombre o está en otra ruta.

Ejemplos:

```python
CSV_PATH = "FakeNews.csv"
CSV_PATH = "data/FakeNews.csv"
CSV_PATH = "/content/FakeNews.csv"
```

In [2]:
CSV_PATH = r"C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\Tema5_Transformers\Listado1\test.xlsx"  # Cambia esta ruta si tu archivo se llama de otra forma

df = pd.read_excel(CSV_PATH)

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nPrimeras filas:")
df.head()

Columnas del dataset:
['ID', 'CATEGORY', 'TOPICS', 'SOURCE', 'HEADLINE', 'TEXT', 'LINK']

Primeras filas:


,ID,CATEGORY,TOPICS,SOURCE,HEADLINE,TEXT,LINK
0,1,True,Covid-19,El Economista,Covid-19: mentiras que matan,El control de la Covid-19 no es sólo un tema d...,https://www.eleconomista.com.mx/opinion/Covid-...
1,2,False,Política,El matinal,El Gobierno podrá acceder a las IPs de los móv...,El Gobierno de Pedro Sánchez y Pablo Iglesias ...,https://www.elmatinal.com/espana-ultima-hora/e...
2,3,True,Política,El País,La comunidad musulmana catalana denuncia a Vox...,Las tres federaciones que agrupan al 90% de la...,https://elpais.com/espana/elecciones-catalanas...
3,4,False,Política,AFPFactual,NaN,Se han dado a conocer los datos electorales pr...,https://perma.cc/GYE6-SPMB
4,5,True,Sociedad,La Republica,El censo poblacional 2018 tendrá un costo de $...,La primera fase del censo será virtual y solo ...,https://www.larepublica.co/economia/el-censo-p...


## 4. Selección de columnas

En este ejercicio vamos a usar:

- El titular (`Headline`).
- El cuerpo de la noticia (`Text`).
- La categoría (`Category`).

Después uniremos `Headline + Text` para que el modelo tenga más información.

In [7]:
HEADLINE_COLUMN = "HEADLINE"
TEXT_COLUMN = "TEXT"
LABEL_COLUMN = "CATEGORY"

required_columns = [HEADLINE_COLUMN, TEXT_COLUMN, LABEL_COLUMN]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(
            f"La columna '{col}' no existe. Columnas disponibles: {df.columns.tolist()}"
        )

print("Columnas necesarias encontradas correctamente.")

Columnas necesarias encontradas correctamente.


## 5. Limpieza de etiquetas

Convertimos la columna `Category` a valores numéricos.

Usaremos esta codificación:

| Etiqueta original | Etiqueta numérica |
|---|---:|
| `true`, `real`, `verdadera` | `0` |
| `false`, `fake`, `falsa` | `1` |

Esto permite usar una salida con una sola neurona y activación `sigmoid`.

In [8]:
print("Valores originales de Category:")
print(df[LABEL_COLUMN].value_counts(dropna=False))

df_clean = df.copy()

df_clean["label"] = (
    df_clean[LABEL_COLUMN]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": 0,
        "real": 0,
        "verdadera": 0,
        "verdadero": 0,
        "0": 0,

        "false": 1,
        "fake": 1,
        "falsa": 1,
        "falso": 1,
        "1": 1
    })
)

print("\nValores NaN después del mapeo:", df_clean["label"].isna().sum())

# Eliminamos filas cuya etiqueta no se haya podido mapear
df_clean = df_clean.dropna(subset=["label"]).copy()
df_clean["label"] = df_clean["label"].astype(int)

print("\nDistribución final de clases:")
print(df_clean["label"].value_counts())

print("\nInterpretación:")
print("0 -> noticia verdadera")
print("1 -> noticia falsa")

Valores originales de Category:
CATEGORY
True     286
False    286
Name: count, dtype: int64

Valores NaN después del mapeo: 0

Distribución final de clases:
label
0    286
1    286
Name: count, dtype: int64

Interpretación:
0 -> noticia verdadera
1 -> noticia falsa


## 6. Creación del texto final

Unimos el titular y el cuerpo de la noticia.

Esto suele ser mejor que usar solo el titular, porque el cuerpo puede contener información útil para detectar si la noticia es falsa o verdadera.

In [10]:
# Rellenamos valores vacíos por cadenas vacías
df_clean[HEADLINE_COLUMN] = df_clean[HEADLINE_COLUMN].fillna("").astype(str)
df_clean[TEXT_COLUMN] = df_clean[TEXT_COLUMN].fillna("").astype(str)

# Creamos una única columna de texto juntando titular + cuerpo
df_clean["full_text"] = (
    df_clean[HEADLINE_COLUMN] + " " + df_clean[TEXT_COLUMN]
)

# Eliminamos espacios sobrantes
df_clean["full_text"] = df_clean["full_text"].str.strip()

# Quitamos filas donde el texto final esté vacío
df_clean = df_clean[df_clean["full_text"] != ""]

texts = df_clean["full_text"].tolist()
y = df_clean["label"].astype(int).values

print("Número total de textos:", len(texts))
print("Número total de etiquetas:", len(y))

print("\nEjemplo de texto:")
print(texts[0][:500])

print("\nEtiqueta del ejemplo:", y[0])

Número total de textos: 572
Número total de etiquetas: 572

Ejemplo de texto:
Covid-19: mentiras que matan El control de la Covid-19 no es sólo un tema de médicos y el resto del personal sanitario y científico. Por desgracia o por fortuna, es un asunto esencialmente político que se decide por hombres y mujeres que se dedican a la política. De las creencias y opiniones de estos últimos, depende el éxito o el fracaso de las acciones que se implementen.

Los éxitos en la toma de decisiones salvan vidas y naciones; obviamente, los errores matan y más si están acompañados de m

Etiqueta del ejemplo: 0


## 7. División en train, dev y test

El enunciado pide partir el dataset en tres partes:

1. Primero separamos en **train** y **dev**.
2. Después dividimos parte de **dev** para crear **test**.

Usaremos aproximadamente:

- `70%` train
- `15%` dev
- `15%` test

Además usamos `stratify` para mantener una distribución de clases parecida en los tres conjuntos.

In [11]:
X_train_texts, X_dev_texts, y_train, y_dev = train_test_split(
    texts,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_dev_texts, X_test_texts, y_dev, y_test = train_test_split(
    X_dev_texts,
    y_dev,
    test_size=0.50,
    random_state=42,
    stratify=y_dev
)

print("Tamaños de los conjuntos:")
print("Train:", len(X_train_texts))
print("Dev:", len(X_dev_texts))
print("Test:", len(X_test_texts))

print("\nDistribución train:", dict(pd.Series(y_train).value_counts()))
print("Distribución dev:", dict(pd.Series(y_dev).value_counts()))
print("Distribución test:", dict(pd.Series(y_test).value_counts()))

Tamaños de los conjuntos:
Train: 400
Dev: 86
Test: 86

Distribución train: {1: np.int64(200), 0: np.int64(200)}
Distribución dev: {1: np.int64(43), 0: np.int64(43)}
Distribución test: {1: np.int64(43), 0: np.int64(43)}


## 8. Tokenización y padding

El Transformer no recibe texto directamente. Primero tenemos que convertir cada noticia en una secuencia de números.

Por ejemplo:

```text
"this news is fake" → [15, 82, 7, 441]
```

Después aplicamos **padding** para que todas las noticias tengan la misma longitud.

Parámetros usados:

| Parámetro | Significado |
|---|---|
| `vocab_size` | Número máximo de tokens distintos que usará el modelo |
| `max_length` | Número máximo de tokens por noticia |
| `embedding_dim` | Tamaño del vector que representa cada token |

In [12]:
vocab_size = 10000
max_length = 100
embedding_dim = 32

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

# Muy importante: ajustamos el tokenizer SOLO con train
tokenizer.fit_on_texts(X_train_texts)

X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
X_dev_seq = tokenizer.texts_to_sequences(X_dev_texts)
X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

X_train = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_dev = pad_sequences(
    X_dev_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

# Conversión a tensores
X_train = tf.convert_to_tensor(X_train, dtype=tf.int32)
X_dev = tf.convert_to_tensor(X_dev, dtype=tf.int32)
X_test = tf.convert_to_tensor(X_test, dtype=tf.int32)

y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
y_dev = tf.convert_to_tensor(y_dev, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

print("Forma de X_train:", X_train.shape)
print("Forma de X_dev:", X_dev.shape)
print("Forma de X_test:", X_test.shape)

Forma de X_train: (400, 100)
Forma de X_dev: (86, 100)
Forma de X_test: (86, 100)


## 9. Modelo A: Transformer sin embeddings posicionales

Esta es la primera versión pedida por el ejercicio.

El modelo usa:

1. `Embedding`: convierte cada token en un vector.
2. `MultiHeadAttention`: aplica self-attention sobre la secuencia.
3. `LayerNormalization`: estabiliza el entrenamiento.
4. `GlobalAveragePooling1D`: resume toda la noticia en un único vector.
5. `Dense(1, sigmoid)`: clasifica entre verdadera o falsa.

En esta versión el modelo sabe **qué tokens aparecen**, pero no recibe explícitamente información sobre **la posición** de cada token.

In [13]:
def create_transformer_without_position(
    vocab_size,
    max_length,
    embedding_dim=32,
    num_heads=2,
    dropout_rate=0.2
):
    inputs = Input(shape=(max_length,), dtype=tf.int32)

    # Embedding de tokens: convierte cada token en un vector de embedding_dim dimensiones
    x = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    )(inputs)

    # Self-attention: query, key y value son la misma secuencia
    attention = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embedding_dim
    )(x, x, x)

    # Conexión residual + normalización
    x = LayerNormalization()(x + attention)

    # Regularización
    x = Dropout(dropout_rate)(x)

    # Convertimos la secuencia completa en un vector único
    x = GlobalAveragePooling1D()(x)

    # Clasificación binaria
    outputs = Dense(1, activation="sigmoid")(x)

    return Model(inputs, outputs)

## 10. Entrenamiento del modelo sin embeddings posicionales

Entrenamos el primer modelo usando el conjunto de entrenamiento y validamos con `dev`.

In [14]:
model_without_position = create_transformer_without_position(
    vocab_size=vocab_size,
    max_length=max_length,
    embedding_dim=embedding_dim
)

model_without_position.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_without_position.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 32)   │    320,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 100, 32)   │      8,416 │ embedding[0][0],  │
│ (MultiHeadAttentio… │                   │            │ embedding[0][0],  │
│                     │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 100, 32)   │          0 │ embedding[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 100, 32)   │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 100, 32)   │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ dropout_1[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         33 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 328,513 (1.25 MB)

 Trainable params: 328,513 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history_without_position = model_without_position.fit(
    X_train,
    y_train,
    validation_data=(X_dev, y_dev),
    epochs=5,
    batch_size=32
)

Epoch 1/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5325 - loss: 0.6904 - val_accuracy: 0.5116 - val_loss: 0.6912
Epoch 2/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8075 - loss: 0.5675 - val_accuracy: 0.5465 - val_loss: 0.6819
Epoch 3/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9050 - loss: 0.4297 - val_accuracy: 0.5465 - val_loss: 0.7239
Epoch 4/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9625 - loss: 0.2384 - val_accuracy: 0.5581 - val_loss: 0.7946
Epoch 5/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9950 - loss: 0.0708 - val_accuracy: 0.5698 - val_loss: 0.9399


## 11. Evaluación del modelo sin embeddings posicionales

Evaluamos el modelo sobre el conjunto de test.

Además de `accuracy`, usamos **Macro F1**, que es útil cuando queremos valorar el rendimiento medio entre clases.

In [16]:
pred_probs_without_position = model_without_position.predict(X_test)
pred_without_position = (pred_probs_without_position > 0.5).astype(int).flatten()

print("Resultados SIN embeddings posicionales:")
print(classification_report(y_test.numpy(), pred_without_position))

acc_without_position = accuracy_score(y_test.numpy(), pred_without_position)
f1_without_position = f1_score(y_test.numpy(), pred_without_position, average="macro")

print("Accuracy sin embeddings posicionales:", acc_without_position)
print("Macro F1 sin embeddings posicionales:", f1_without_position)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Resultados SIN embeddings posicionales:
              precision    recall  f1-score   support

         0.0       0.67      0.47      0.55        43
         1.0       0.59      0.77      0.67        43

    accuracy                           0.62        86
   macro avg       0.63      0.62      0.61        86
weighted avg       0.63      0.62      0.61        86

Accuracy sin embeddings posicionales: 0.6162790697674418
Macro F1 sin embeddings posicionales: 0.6073059360730593


## 12. Modelo B: Transformer con embeddings posicionales

Ahora añadimos embeddings posicionales.

Un Transformer con self-attention no incorpora de forma natural el orden de los tokens. Por eso, se suma al embedding de cada token un vector que representa su posición dentro de la secuencia.

La idea es:

```text
representación final = embedding del token + embedding de posición
```

Así el modelo sabe no solo **qué palabra aparece**, sino también **dónde aparece**.

In [17]:
def create_transformer_with_position(
    vocab_size,
    max_length,
    embedding_dim=32,
    num_heads=2,
    dropout_rate=0.2
):


    inputs = Input(shape=(max_length,), dtype=tf.int32)

    # Embedding de tokens
    token_embeddings = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    )(inputs)

    # Embedding de posición
    position_embeddings = keras_nlp.layers.PositionEmbedding(
        sequence_length=max_length
    )(token_embeddings)

    # Sumamos información del token + información de posición
    x = token_embeddings + position_embeddings

    # Self-attention
    attention = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embedding_dim
    )(x, x, x)

    # Conexión residual + normalización
    x = LayerNormalization()(x + attention)

    # Regularización
    x = Dropout(dropout_rate)(x)

    # Vector único para toda la noticia
    x = GlobalAveragePooling1D()(x)

    # Clasificación binaria
    outputs = Dense(1, activation="sigmoid")(x)

    return Model(inputs, outputs)

## 13. Entrenamiento del modelo con embeddings posicionales

Entrenamos el segundo modelo con la misma división de datos para que la comparación sea justa.

In [18]:
model_with_position = create_transformer_with_position(
    vocab_size=vocab_size,
    max_length=max_length,
    embedding_dim=embedding_dim
)

model_with_position.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_with_position.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 100, 32)   │    320,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, 100, 32)   │      3,200 │ embedding_1[0][0] │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 100, 32)   │          0 │ embedding_1[0][0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 100, 32)   │      8,416 │ add_1[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_1[0][0],      │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 100, 32)   │          0 │ add_1[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 100, 32)   │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 100, 32)   │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ dropout_3[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         33 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 331,713 (1.27 MB)

 Trainable params: 331,713 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history_with_position = model_with_position.fit(
    X_train,
    y_train,
    validation_data=(X_dev, y_dev),
    epochs=5,
    batch_size=32
)

Epoch 1/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.4975 - loss: 0.6961 - val_accuracy: 0.5465 - val_loss: 0.6893
Epoch 2/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8025 - loss: 0.6370 - val_accuracy: 0.5698 - val_loss: 0.6850
Epoch 3/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8475 - loss: 0.5629 - val_accuracy: 0.5000 - val_loss: 0.6939
Epoch 4/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8925 - loss: 0.4231 - val_accuracy: 0.4884 - val_loss: 0.7693
Epoch 5/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9525 - loss: 0.2240 - val_accuracy: 0.5930 - val_loss: 0.8203


## 14. Evaluación del modelo con embeddings posicionales

Evaluamos el segundo modelo sobre el conjunto de test.

In [20]:
pred_probs_with_position = model_with_position.predict(X_test)
pred_with_position = (pred_probs_with_position > 0.5).astype(int).flatten()

print("Resultados CON embeddings posicionales:")
print(classification_report(y_test.numpy(), pred_with_position))

acc_with_position = accuracy_score(y_test.numpy(), pred_with_position)
f1_with_position = f1_score(y_test.numpy(), pred_with_position, average="macro")

print("Accuracy con embeddings posicionales:", acc_with_position)
print("Macro F1 con embeddings posicionales:", f1_with_position)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Resultados CON embeddings posicionales:
              precision    recall  f1-score   support

         0.0       0.62      0.47      0.53        43
         1.0       0.57      0.72      0.64        43

    accuracy                           0.59        86
   macro avg       0.60      0.59      0.59        86
weighted avg       0.60      0.59      0.59        86

Accuracy con embeddings posicionales: 0.5930232558139535
Macro F1 con embeddings posicionales: 0.586254295532646


## 15. Comparación final

Comparamos ambos modelos en una tabla.

Lo más importante del ejercicio no es solo obtener un número alto, sino analizar si añadir información posicional mejora o no el rendimiento.

In [21]:
results = pd.DataFrame({
    "Modelo": [
        "Transformer sin embeddings posicionales",
        "Transformer con embeddings posicionales"
    ],
    "Accuracy": [
        acc_without_position,
        acc_with_position
    ],
    "Macro F1": [
        f1_without_position,
        f1_with_position
    ]
})

results

,Modelo,Accuracy,Macro F1
0,Transformer sin embeddings posicionales,0.616279,0.607306
1,Transformer con embeddings posicionales,0.593023,0.586254


## 16. Conclusión automática

Esta celda genera una conclusión básica en función del valor de **Macro F1**.

Después puedes copiarla y adaptarla para la memoria de la práctica.

## 17. Texto de conclusión para la memoria

Puedes usar este párrafo como base, sustituyendo los valores `X` e `Y` por los resultados obtenidos:

> En este ejercicio se ha entrenado un modelo Transformer básico para clasificar noticias falsas. Primero se ha utilizado una versión sin embeddings posicionales y después una versión con embeddings posicionales mediante la capa `PositionEmbedding` de `keras-nlp`. La diferencia principal entre ambos modelos es que el segundo incorpora información sobre la posición de cada token dentro de la secuencia. Tras evaluar ambos modelos sobre el conjunto de test, el modelo sin embeddings posicionales obtuvo un Macro F1 de `X`, mientras que el modelo con embeddings posicionales obtuvo un Macro F1 de `Y`. Por tanto, en este caso, el uso de embeddings posicionales `[mejora/no mejora]` el rendimiento. Aunque los embeddings posicionales suelen ser importantes en Transformers, los resultados pueden variar en función del tamaño del dataset, el número de épocas y la configuración del modelo.